In [158]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pylab import rcParams
rcParams['figure.figsize'] = 15, 6

import openturns as ot
import openturns.viewer as otv
ot.Log.Show(ot.Log.NONE)

import functions_probabilistic as fp

In [159]:
# Constants
global gamma_w, gamma_s, d70_m, L, H, tan_theta
global eta, nu, g, d, rho_sub, C_clay, C_sand
global Lv_clay, Lv_sand

gamma_w = 1025 * 9.81 # unit weight of the water
gamma_s = 16500 # unit weight of the submerged particle

# d70 = 2.8e-4 # 70%-fractile of grain size distribution
d70_m = 2.08e-4 # Reference value of 70%-fractile of grain size dis-tribution
L = 45.4 + 9 # piping length (adjusted to new situation)
H = 5.3 # water level at the foreside of the dike
tan_theta = np.tan(35 * (np.pi/180)) # slope of the dike
# k = 7.52e-4 # hydraulic conductivity of the auqifer
# D = 6.0 # thickness of the aquifer
eta = 0.25 # Drag factor coefficient 
# m_p = 1 # Model factor piping
nu = 1.33e-6 # Kinematic viscosity 
g = 9.81 # Gravitational acceleration
# h_b = 0 # water level on the hinter side of the dike
d = 2.5 - 0.5 # impermeable clay layer at the sand boil exit point (adjusted)
rho_sub = 1.25 # factor of safety

C_clay = 8.5
C_sand = 6

Lv_clay = 2.5
Lv_sand = 6




In [160]:
# Variables
d70 = ot.LogNormal(2.8e-4, 0.12)
k = ot.LogNormal(7.52e-4, 0.50)
D = ot.LogNormal(6.0, 0.25)
m_p = ot.Normal(1.0, 0.12)
h_p = ot.Normal(-0.5, 0.1)

x = (d70, k, D, m_p, h_p)

descriptions = ["grain size d70",
                "hydraulic conductivity k",
                "thickness of the aquifer D",
                "model factor m_p",
                "phreatic level hinterland h_p"]

In [161]:
def LSF(x):
    d70, k, D, m_p, h_p = x
    F_R = (eta*((gamma_s/gamma_w)-1)*tan_theta)
    F_S = ((d70_m / (((nu*k*L)/g)**(1/3))) * ((d70/d70_m)**0.4))
    step_1 = ((D/L)**2.8)-1
    step_2 = (0.28/(step_1))+0.04
    F_G = (0.91*((D/L)**step_2))
    H_c = (((L/3) + (2 * Lv_sand)) / C_sand) + ((2 * Lv_clay) / C_clay)
    # H_c = m_p * F_R * F_S * F_G * L / rho_sub
    Z = (H_c) - (H - h_p - d*0.3)
    return [Z]


In [162]:
LSF((2.8e-4, 7.52e-4, 6.0, 1.0, -0.5))

[0.4104575163398687]

In [163]:
fp.input_OpenTurns(x, descriptions, LSF, 0)

In [164]:
result, x_star, u_star, pf_FORM, beta = fp.run_FORM_analysis()

The FORM analysis took 0.032 seconds
FORM result, pf = 0.0000
FORM result, beta = 4.104

The design point in the u space:  [-8.01428e-06,-4.47622e-06,-3.63005e-05,-1.80827e-07,-4.10358]
The design point in the x space:  [1.00028,1.00075,403.425,1,-0.910358]


In [165]:
it = 0
maxit = 100
Lv_clay_it = Lv_clay
while pf_FORM > 1.4e-6 and it < maxit:
    Lv_clay += 0.1
    it += 1
    fp.input_OpenTurns(x, descriptions, LSF, 0)
    result, x_star, u_star, pf_FORM, beta = fp.run_FORM_analysis(printing=False)
    print(f"Iteration {it}: pf_FORM = {pf_FORM}, Lv_clay = {Lv_clay}")



Iteration 1: pf_FORM = 7.16088360977501e-06, Lv_clay = 2.6
Iteration 2: pf_FORM = 2.390632965589033e-06, Lv_clay = 2.7
Iteration 3: pf_FORM = 7.567022746494506e-07, Lv_clay = 2.8000000000000003
